# Documented English edition

This notebook is the reviewed English edition of `exercicios_computacao_quantica/portas_transversais_injecao_erros.ipynb`. The original file, metadata, and historical outputs are preserved under `codes_obsolete/`. Stored outputs were cleared from this edition; execute the cells sequentially with the declared kernel.


In [ ]:
# Purpose: Import the numerical, visualization, and quantum-computing libraries used below.
# ============================================================
# Explanation translated; consult the archived notebook for the original wording.
# ============================================================
#
# Explanation translated; consult the archived notebook for the original wording.
# pip install qiskit qiskit-aer matplotlib pylatexenc
#
# Explanation translated; consult the archived notebook for the original wording.
# Explanation translated; consult the archived notebook for the original wording.
# Explanation translated; consult the archived notebook for the original wording.
# 3. CNOT transversal entre blocos codificados;
# 4. Shor code de 9 qubits;
# Explanation translated; consult the archived notebook for the original wording.
# 6. Leitura de estabilizadores do Shor code;
# Explanation translated; consult the archived notebook for the original wording.
#
# ============================================================

from collections import Counter

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit.quantum_info import Statevector, SparsePauliOp


# ============================================================
# Explanation translated; consult the archived notebook for the original wording.
# ============================================================

def simulate_counts(qc, shots=2048, noise_model=None):
    """English documentation for this computational helper."""

    simulator = AerSimulator(noise_model=noise_model)
    compiled = transpile(qc, simulator)
    result = simulator.run(compiled, shots=shots).result()

    return result.get_counts()


def add_all_measurements(qc):
    """English documentation for this computational helper."""

    measured = QuantumCircuit(qc.num_qubits, qc.num_qubits)
    measured.compose(qc, qubits=range(qc.num_qubits), inplace=True)
    measured.measure(range(qc.num_qubits), range(qc.num_qubits))

    return measured


def classical_bit(bitstring, index):
    """English documentation for this computational helper."""

    clean = bitstring.replace(" ", "")
    return int(clean[::-1][index])


def top_counts(counts, k=10):
    """
    Retorna apenas os k resultados mais frequentes.
    """

    return dict(
        sorted(counts.items(), key=lambda item: item[1], reverse=True)[:k]
    )


def inject_pauli_error(qc, qubit, pauli):
    """English documentation for this computational helper."""

    if pauli is None:
        return

    pauli = pauli.upper()

    if pauli == "X":
        qc.x(qubit)
    elif pauli == "Y":
        qc.y(qubit)
    elif pauli == "Z":
        qc.z(qubit)
    else:
        raise ValueErrorr("O erro deve ser 'X', 'Y', 'Z' ou None.")


def build_depolarizing_noise_model(p1=0.001, p2=0.01):
    """English documentation for this computational helper."""

    noise_model = NoiseModel()

    one_qubit_error = depolarizing_error(p1, 1)
    two_qubit_error = depolarizing_error(p2, 2)

    noise_model.add_all_qubit_quantum_error(
        one_qubit_error,
        ["x", "y", "z", "h"]
    )

    noise_model.add_all_qubit_quantum_error(
        two_qubit_error,
        ["cx"]
    )

    return noise_model


# ============================================================
# Explanation translated; consult the archived notebook for the original wording.
# ============================================================

def encode_repetition_bitflip(qc, data_qubits=(0, 1, 2), logical_state="0"):
    """English documentation for this computational helper."""

    q0, q1, q2 = data_qubits

    if logical_state == "1":
        qc.x(q0)
    elif logical_state == "0":
        pass
    else:
        raise ValueErrorr("logical_state deve ser '0' ou '1'.")

    qc.cx(q0, q1)
    qc.cx(q0, q2)


def build_repetition_syndrome_circuit(
    logical_state="0",
    error_qubit=None,
    pauli_error=None
):
    """English documentation for this computational helper."""

    qc = QuantumCircuit(5, 5)

    encode_repetition_bitflip(
        qc,
        data_qubits=(0, 1, 2),
        logical_state=logical_state
    )

    if error_qubit is not None:
        inject_pauli_error(qc, error_qubit, pauli_error)

    # Explanation translated; consult the archived notebook for the original wording.
    qc.cx(0, 3)
    qc.cx(1, 3)

    # Explanation translated; consult the archived notebook for the original wording.
    qc.cx(1, 4)
    qc.cx(2, 4)

    qc.measure([0, 1, 2, 3, 4], [0, 1, 2, 3, 4])

    return qc


def summarize_repetition_syndrome(counts):
    """English documentation for this computational helper."""

    syndrome_counts = Counter()
    logical_counts = Counter()

    for bitstring, freq in counts.items():
        b0 = classical_bit(bitstring, 0)
        b1 = classical_bit(bitstring, 1)
        b2 = classical_bit(bitstring, 2)
        s01 = classical_bit(bitstring, 3)
        s12 = classical_bit(bitstring, 4)

        syndrome = f"{s01}{s12}"
        syndrome_counts[syndrome] += freq

        logical_value = 1 if (b0 + b1 + b2) >= 2 else 0
        logical_counts[str(logical_value)] += freq

    return dict(syndrome_counts), dict(logical_counts)


def interpret_repetition_syndrome(syndrome):
    """English documentation for this computational helper."""

    table = {
        "00": "English diagnostic label",
        "10": "English diagnostic label",
        "11": "English diagnostic label",
        "01": "English diagnostic label",
    }

    return table.get(syndrome, "syndrome desconhecida")


# ============================================================
# Explanation translated; consult the archived notebook for the original wording.
# ============================================================

def build_repetition_transversal_cnot(
    control_state="1",
    target_state="0",
    error_qubit=None,
    pauli_error=None
):
    """English documentation for this computational helper."""

    qc = QuantumCircuit(6, 6)

    encode_repetition_bitflip(
        qc,
        data_qubits=(0, 1, 2),
        logical_state=control_state
    )

    encode_repetition_bitflip(
        qc,
        data_qubits=(3, 4, 5),
        logical_state=target_state
    )

    if error_qubit is not None:
        inject_pauli_error(qc, error_qubit, pauli_error)

    # Explanation translated; consult the archived notebook for the original wording.
    qc.cx(0, 3)
    qc.cx(1, 4)
    qc.cx(2, 5)

    qc.measure(range(6), range(6))

    return qc


def summarize_two_repetition_blocks(counts):
    """English documentation for this computational helper."""

    logical_counts = Counter()

    for bitstring, freq in counts.items():
        block_a = [
            classical_bit(bitstring, 0),
            classical_bit(bitstring, 1),
            classical_bit(bitstring, 2),
        ]

        block_b = [
            classical_bit(bitstring, 3),
            classical_bit(bitstring, 4),
            classical_bit(bitstring, 5),
        ]

        logical_a = 1 if sum(block_a) >= 2 else 0
        logical_b = 1 if sum(block_b) >= 2 else 0

        logical_counts[f"{logical_a}{logical_b}"] += freq

    return dict(logical_counts)


# ============================================================
# Explanation translated; consult the archived notebook for the original wording.
# ============================================================

def encode_shor_block(qc, offset=0, logical_state="0"):
    """English documentation for this computational helper."""

    q = [offset + i for i in range(9)]

    if logical_state == "0":
        pass
    elif logical_state == "1":
        qc.x(q[0])
    elif logical_state == "+":
        qc.h(q[0])
    elif logical_state == "-":
        qc.x(q[0])
        qc.h(q[0])
    else:
        raise ValueErrorr("logical_state deve ser '0', '1', '+' ou '-'.")

    # Explanation translated; consult the archived notebook for the original wording.
    qc.cx(q[0], q[3])
    qc.cx(q[0], q[6])

    qc.h(q[0])
    qc.h(q[3])
    qc.h(q[6])

    # Explanation translated; consult the archived notebook for the original wording.
    qc.cx(q[0], q[1])
    qc.cx(q[0], q[2])

    qc.cx(q[3], q[4])
    qc.cx(q[3], q[5])

    qc.cx(q[6], q[7])
    qc.cx(q[6], q[8])


def build_shor_single_block(
    logical_state="0",
    error_qubit=None,
    pauli_error=None,
    transversal_gate=None
):
    """English documentation for this computational helper."""

    qc = QuantumCircuit(9)

    encode_shor_block(qc, offset=0, logical_state=logical_state)

    if error_qubit is not None:
        inject_pauli_error(qc, error_qubit, pauli_error)

    if transversal_gate is not None:
        transversal_gate = transversal_gate.upper()

        if transversal_gate == "X":
            for i in range(9):
                qc.x(i)

        elif transversal_gate == "Z":
            for i in range(9):
                qc.z(i)

        else:
            raise ValueErrorr("transversal_gate deve ser None, 'X' ou 'Z'.")

    return qc


# ============================================================
# Explanation translated; consult the archived notebook for the original wording.
# ============================================================

SHOR_STABILIZERS = [
    ("Z0 Z1", {0: "Z", 1: "Z"}),
    ("Z1 Z2", {1: "Z", 2: "Z"}),
    ("Z3 Z4", {3: "Z", 4: "Z"}),
    ("Z4 Z5", {4: "Z", 5: "Z"}),
    ("Z6 Z7", {6: "Z", 7: "Z"}),
    ("Z7 Z8", {7: "Z", 8: "Z"}),
    ("X0 X1 X2 X3 X4 X5", {0: "X", 1: "X", 2: "X", 3: "X", 4: "X", 5: "X"}),
    ("X3 X4 X5 X6 X7 X8", {3: "X", 4: "X", 5: "X", 6: "X", 7: "X", 8: "X"}),
]


def pauli_operator(num_qubits, local_ops):
    """English documentation for this computational helper."""

    label = ["I"] * num_qubits

    for qubit, pauli in local_ops.items():
        label[num_qubits - 1 - qubit] = pauli

    return SparsePauliOp("".join(label))


def shor_stabilizer_expectations(qc, offset=0):
    """English documentation for this computational helper."""

    sv = Statevector.from_instruction(qc)
    table = []

    for name, ops in SHOR_STABILIZERS:
        shifted_ops = {offset + qubit: pauli for qubit, pauli in ops.items()}
        op = pauli_operator(qc.num_qubits, shifted_ops)

        value = float(sv.expectation_value(op).real)

        if value > 0.5:
            sign = "+1"
        elif value < -0.5:
            sign = "-1"
        else:
            sign = "0"

        table.append((name, sign, round(value, 6)))

    return table


def print_stabilizer_table(title, table):
    """
    Imprime tabela de estabilizadores.
    """

    print("\n" + title)
    print("-" * len(title))

    for name, sign, value in table:
        print(f"{name:25s}  sinal = {sign:>2s}   valor = {value: .6f}")


# ============================================================
# PARTE 5 — CNOT TRANSVERSAL ENTRE DOIS BLOCOS DE SHOR
# ============================================================

def build_shor_transversal_cnot(
    control_state="1",
    target_state="0",
    error_block=None,
    error_position=None,
    pauli_error=None
):
    """English documentation for this computational helper."""

    qc = QuantumCircuit(18)

    encode_shor_block(qc, offset=0, logical_state=control_state)
    encode_shor_block(qc, offset=9, logical_state=target_state)

    if error_block is not None:
        if error_position is None:
            raise ValueErrorr("Defina error_position entre 0 e 8.")

        if error_block == "control":
            physical_qubit = error_position
        elif error_block == "target":
            physical_qubit = 9 + error_position
        else:
            raise ValueErrorr("error_block deve ser None, 'control' ou 'target'.")

        inject_pauli_error(qc, physical_qubit, pauli_error)

    # CNOT transversal entre os blocos
    for i in range(9):
        qc.cx(i, 9 + i)

    return qc


# ============================================================
# Explanation translated; consult the archived notebook for the original wording.
# ============================================================

print("\n============================================================")
print("English diagnostic label")
print("============================================================")

qc_rep_no_error = build_repetition_syndrome_circuit(
    logical_state="0",
    error_qubit=None,
    pauli_error=None
)

counts_rep_no_error = simulate_counts(qc_rep_no_error)
syndrome_no_error, logical_no_error = summarize_repetition_syndrome(counts_rep_no_error)

print("Counts:", counts_rep_no_error)
print("English diagnostic label", syndrome_no_error)
print("English diagnostic label", logical_no_error)

for syndrome in syndrome_no_error:
    print(f"English diagnostic label")


print("\n============================================================")
print("English diagnostic label")
print("============================================================")

qc_rep_x_error = build_repetition_syndrome_circuit(
    logical_state="0",
    error_qubit=1,
    pauli_error="X"
)

counts_rep_x_error = simulate_counts(qc_rep_x_error)
syndrome_x_error, logical_x_error = summarize_repetition_syndrome(counts_rep_x_error)

print("Counts:", counts_rep_x_error)
print("English diagnostic label", syndrome_x_error)
print("English diagnostic label", logical_x_error)

for syndrome in syndrome_x_error:
    print(f"English diagnostic label")


print("\n============================================================")
print("English diagnostic label")
print("============================================================")

qc_rep_tcnot = build_repetition_transversal_cnot(
    control_state="1",
    target_state="0",
    error_qubit=None,
    pauli_error=None
)

counts_rep_tcnot = simulate_counts(qc_rep_tcnot)
logical_rep_tcnot = summarize_two_repetition_blocks(counts_rep_tcnot)

print("Counts:", counts_rep_tcnot)
print("English diagnostic label", logical_rep_tcnot)


print("\n============================================================")
print("English diagnostic label")
print("============================================================")

qc_rep_tcnot_error = build_repetition_transversal_cnot(
    control_state="1",
    target_state="0",
    error_qubit=1,
    pauli_error="X"
)

counts_rep_tcnot_error = simulate_counts(qc_rep_tcnot_error)
logical_rep_tcnot_error = summarize_two_repetition_blocks(counts_rep_tcnot_error)

print("Counts:", counts_rep_tcnot_error)
print("English diagnostic label", logical_rep_tcnot_error)


print("\n============================================================")
print("English diagnostic label")
print("============================================================")

qc_shor_no_error = build_shor_single_block(
    logical_state="0",
    error_qubit=None,
    pauli_error=None
)

table_shor_no_error = shor_stabilizer_expectations(qc_shor_no_error)
print_stabilizer_table("Estabilizadores do Shor sem erro", table_shor_no_error)

counts_shor_no_error = simulate_counts(add_all_measurements(qc_shor_no_error))
print("Top counts:", top_counts(counts_shor_no_error))


print("\n============================================================")
print("English diagnostic label")
print("============================================================")

qc_shor_x_error = build_shor_single_block(
    logical_state="0",
    error_qubit=4,
    pauli_error="X"
)

table_shor_x_error = shor_stabilizer_expectations(qc_shor_x_error)
print_stabilizer_table("English diagnostic label", table_shor_x_error)

counts_shor_x_error = simulate_counts(add_all_measurements(qc_shor_x_error))
print("Top counts:", top_counts(counts_shor_x_error))


print("\n============================================================")
print("English diagnostic label")
print("============================================================")

qc_shor_z_error = build_shor_single_block(
    logical_state="0",
    error_qubit=4,
    pauli_error="Z"
)

table_shor_z_error = shor_stabilizer_expectations(qc_shor_z_error)
print_stabilizer_table("English diagnostic label", table_shor_z_error)

counts_shor_z_error = simulate_counts(add_all_measurements(qc_shor_z_error))
print("Top counts:", top_counts(counts_shor_z_error))


print("\n============================================================")
print("English diagnostic label")
print("============================================================")

qc_shor_y_error = build_shor_single_block(
    logical_state="0",
    error_qubit=4,
    pauli_error="Y"
)

table_shor_y_error = shor_stabilizer_expectations(qc_shor_y_error)
print_stabilizer_table("English diagnostic label", table_shor_y_error)

counts_shor_y_error = simulate_counts(add_all_measurements(qc_shor_y_error))
print("Top counts:", top_counts(counts_shor_y_error))


print("\n============================================================")
print("EXPERIMENTO 9 — CNOT TRANSVERSAL ENTRE DOIS BLOCOS DE SHOR")
print("============================================================")

qc_shor_tcnot = build_shor_transversal_cnot(
    control_state="1",
    target_state="0",
    error_block=None,
    error_position=None,
    pauli_error=None
)

table_control = shor_stabilizer_expectations(qc_shor_tcnot, offset=0)
table_target = shor_stabilizer_expectations(qc_shor_tcnot, offset=9)

print_stabilizer_table("English diagnostic label", table_control)
print_stabilizer_table("English diagnostic label", table_target)

counts_shor_tcnot = simulate_counts(add_all_measurements(qc_shor_tcnot))
print("Top counts:", top_counts(counts_shor_tcnot))


print("\n============================================================")
print("English diagnostic label")
print("============================================================")

qc_shor_tcnot_error = build_shor_transversal_cnot(
    control_state="1",
    target_state="0",
    error_block="control",
    error_position=2,
    pauli_error="X"
)

table_control_error = shor_stabilizer_expectations(qc_shor_tcnot_error, offset=0)
table_target_error = shor_stabilizer_expectations(qc_shor_tcnot_error, offset=9)

print_stabilizer_table("English diagnostic label", table_control_error)
print_stabilizer_table("English diagnostic label", table_target_error)

counts_shor_tcnot_error = simulate_counts(add_all_measurements(qc_shor_tcnot_error))
print("Top counts:", top_counts(counts_shor_tcnot_error))


print("\n============================================================")
print("English diagnostic label")
print("============================================================")

noise_model = build_depolarizing_noise_model(
    p1=0.001,
    p2=0.01
)

qc_noisy = add_all_measurements(qc_shor_tcnot)

counts_noisy = simulate_counts(
    qc_noisy,
    shots=4096,
    noise_model=noise_model
)

print("English diagnostic label", top_counts(counts_noisy, k=15))


print("\n============================================================")
print("English diagnostic label")
print("============================================================")

## Dependencies

Import the numerical, visualization, and quantum-computing libraries used below.


In [ ]:
# Purpose: Import the numerical, visualization, and quantum-computing libraries used below.
# ============================================================
# Explanation translated; consult the archived notebook for the original wording.
# Explanation translated; consult the archived notebook for the original wording.
# ============================================================
#
# Explanation translated; consult the archived notebook for the original wording.
# pip install qiskit qiskit-aer matplotlib pylatexenc
#
# Objetivos:
# Explanation translated; consult the archived notebook for the original wording.
# Explanation translated; consult the archived notebook for the original wording.
# Explanation translated; consult the archived notebook for the original wording.
# Explanation translated; consult the archived notebook for the original wording.
# Explanation translated; consult the archived notebook for the original wording.
#
# ============================================================

from collections import Counter

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, ReadoutErrorr

try:
    import matplotlib.pyplot as plt
except ImportErrorr:
    plt = None


# ============================================================
# Explanation translated; consult the archived notebook for the original wording.
# ============================================================

SHOTS = 4096
SEED = 1234

N_ROUNDS = 5
READOUT_ERROR_PROBABILITY = 0.15

MAKE_PLOTS = True


# ============================================================
# Explanation translated; consult the archived notebook for the original wording.
# ============================================================

def simulate_counts(qc, shots=SHOTS, noise_model=None, seed=SEED):
    """English documentation for this computational helper."""

    simulator = AerSimulator(noise_model=noise_model)

    compiled = transpile(
        qc,
        simulator,
        optimization_level=0
    )

    job = simulator.run(
        compiled,
        shots=shots,
        seed_simulator=seed
    )

    result = job.result()
    return result.get_counts()


def inject_pauli_error(qc, qubit, pauli):
    """English documentation for this computational helper."""

    if pauli is None:
        return

    pauli = pauli.upper()

    if pauli == "X":
        qc.x(qubit)
    elif pauli == "Y":
        qc.y(qubit)
    elif pauli == "Z":
        qc.z(qubit)
    else:
        raise ValueErrorr("pauli deve ser 'X', 'Y', 'Z' ou None.")


def classical_bit(bitstring, cindex):
    """English documentation for this computational helper."""

    clean = bitstring.replace(" ", "")
    return int(clean[::-1][cindex])


def build_ancilla_readout_noise_model(ancilla_qubits, p_readout=0.10):
    """English documentation for this computational helper."""

    noise_model = NoiseModel()

    readout_error = ReadoutErrorr([
        [1.0 - p_readout, p_readout],
        [p_readout, 1.0 - p_readout]
    ])

    for qubit in ancilla_qubits:
        noise_model.add_readout_error(readout_error, [qubit])

    return noise_model


def top_counts(counts, k=10):
    """
    Retorna os k resultados mais frequentes.
    """

    return dict(
        sorted(
            counts.items(),
            key=lambda item: item[1],
            reverse=True
        )[:k]
    )


def most_likely(counts):
    """English documentation for this computational helper."""

    if len(counts) == 0:
        return None, 0

    return max(counts.items(), key=lambda item: item[1])


def plot_distribution(counts, title, filename=None, top_k=15):
    """English documentation for this computational helper."""

    if plt is None:
        print("English diagnostic label")
        return

    data = top_counts(counts, k=top_k)

    labels = list(data.keys())
    values = list(data.values())

    plt.figure(figsize=(10, 5))
    plt.bar(labels, values)
    plt.title(title)
    plt.xlabel("Resultado")
    plt.ylabel("English diagnostic label")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    if filename is not None:
        plt.savefig(filename, dpi=200, bbox_inches="tight")

    plt.show()


def print_distribution(title, counts, k=10):
    """English documentation for this computational helper."""

    print("\n" + title)
    print("-" * len(title))

    for key, value in top_counts(counts, k=k).items():
        print(f"{key:>12s} : {value}")


def syndrome_from_bitstring(bitstring, n_stabilizers, round_index):
    """English documentation for this computational helper."""

    start = round_index * n_stabilizers

    bits = [
        str(classical_bit(bitstring, start + s))
        for s in range(n_stabilizers)
    ]

    return "".join(bits)


def single_round_syndrome_counts(counts, n_stabilizers, round_index=0):
    """English documentation for this computational helper."""

    syndrome_counts = Counter()

    for bitstring, frequency in counts.items():
        syndrome = syndrome_from_bitstring(
            bitstring,
            n_stabilizers=n_stabilizers,
            round_index=round_index
        )

        syndrome_counts[syndrome] += frequency

    return dict(syndrome_counts)


def majority_syndrome_from_bitstring(
    bitstring,
    n_stabilizers,
    n_rounds,
    tie_value=0
):
    """English documentation for this computational helper."""

    majority_bits = []

    for s in range(n_stabilizers):
        votes = 0

        for r in range(n_rounds):
            cindex = r * n_stabilizers + s
            votes += classical_bit(bitstring, cindex)

        if votes > n_rounds / 2:
            bit = 1
        elif votes < n_rounds / 2:
            bit = 0
        else:
            bit = tie_value

        majority_bits.append(str(bit))

    return "".join(majority_bits)


def majority_syndrome_counts(counts, n_stabilizers, n_rounds, tie_value=0):
    """English documentation for this computational helper."""

    voted_counts = Counter()

    for bitstring, frequency in counts.items():
        syndrome = majority_syndrome_from_bitstring(
            bitstring=bitstring,
            n_stabilizers=n_stabilizers,
            n_rounds=n_rounds,
            tie_value=tie_value
        )

        voted_counts[syndrome] += frequency

    return dict(voted_counts)


# ============================================================
# Explanation translated; consult the archived notebook for the original wording.
# ============================================================

def encode_repetition_bitflip(qc, data_qubits=(0, 1, 2), logical_state="0"):
    """English documentation for this computational helper."""

    q0, q1, q2 = data_qubits

    if logical_state == "0":
        pass
    elif logical_state == "1":
        qc.x(q0)
    else:
        raise ValueErrorr("logical_state deve ser '0' ou '1'.")

    qc.cx(q0, q1)
    qc.cx(q0, q2)


def repetition_ancilla_qubits():
    """English documentation for this computational helper."""

    return [3, 4]


def build_repetition_syndrome_circuit(
    logical_state="0",
    error_qubit=None,
    pauli_error=None,
    n_rounds=1
):
    """English documentation for this computational helper."""

    n_data = 3
    n_ancillas = 2
    n_stabilizers = 2
    n_classical = n_rounds * n_stabilizers

    qc = QuantumCircuit(n_data + n_ancillas, n_classical)

    encode_repetition_bitflip(
        qc,
        data_qubits=(0, 1, 2),
        logical_state=logical_state
    )

    if error_qubit is not None:
        inject_pauli_error(qc, error_qubit, pauli_error)

    anc_s0 = 3
    anc_s1 = 4

    for r in range(n_rounds):
        c_s0 = r * n_stabilizers + 0
        c_s1 = r * n_stabilizers + 1

        # Mede S0 = Z0 Z1
        qc.cx(0, anc_s0)
        qc.cx(1, anc_s0)
        qc.measure(anc_s0, c_s0)

        # Mede S1 = Z1 Z2
        qc.cx(1, anc_s1)
        qc.cx(2, anc_s1)
        qc.measure(anc_s1, c_s1)

        if r < n_rounds - 1:
            qc.reset(anc_s0)
            qc.reset(anc_s1)

        qc.barrier()

    return qc


def interpret_repetition_syndrome(syndrome):
    """English documentation for this computational helper."""

    table = {
        "00": "English diagnostic label",
        "10": "English diagnostic label",
        "11": "English diagnostic label",
        "01": "English diagnostic label",
    }

    return table.get(syndrome, "syndrome desconhecida")


# ============================================================
# Explanation translated; consult the archived notebook for the original wording.
# ============================================================

def encode_shor_block(qc, offset=0, logical_state="0"):
    """English documentation for this computational helper."""

    q = [offset + i for i in range(9)]

    if logical_state == "0":
        pass
    elif logical_state == "1":
        qc.x(q[0])
    elif logical_state == "+":
        qc.h(q[0])
    elif logical_state == "-":
        qc.x(q[0])
        qc.h(q[0])
    else:
        raise ValueErrorr("logical_state deve ser '0', '1', '+' ou '-'.")

    # Camada contra phase flip
    qc.cx(q[0], q[3])
    qc.cx(q[0], q[6])

    qc.h(q[0])
    qc.h(q[3])
    qc.h(q[6])

    # Camada contra bit flip
    qc.cx(q[0], q[1])
    qc.cx(q[0], q[2])

    qc.cx(q[3], q[4])
    qc.cx(q[3], q[5])

    qc.cx(q[6], q[7])
    qc.cx(q[6], q[8])


SHOR_STABILIZERS = [
    ("Z0Z1", "Z", [0, 1]),
    ("Z1Z2", "Z", [1, 2]),
    ("Z3Z4", "Z", [3, 4]),
    ("Z4Z5", "Z", [4, 5]),
    ("Z6Z7", "Z", [6, 7]),
    ("Z7Z8", "Z", [7, 8]),
    ("X0X1X2X3X4X5", "X", [0, 1, 2, 3, 4, 5]),
    ("X3X4X5X6X7X8", "X", [3, 4, 5, 6, 7, 8]),
]


def measure_pauli_product(qc, data_qubits, pauli, ancilla, classical_bit_index):
    """English documentation for this computational helper."""

    pauli = pauli.upper()

    if pauli == "Z":
        for q in data_qubits:
            qc.cx(q, ancilla)

        qc.measure(ancilla, classical_bit_index)

    elif pauli == "X":
        for q in data_qubits:
            qc.h(q)

        for q in data_qubits:
            qc.cx(q, ancilla)

        for q in data_qubits:
            qc.h(q)

        qc.measure(ancilla, classical_bit_index)

    else:
        raise ValueErrorr("pauli deve ser 'X' ou 'Z'.")


def shor_ancilla_qubits():
    """English documentation for this computational helper."""

    return list(range(9, 17))


def build_shor_syndrome_circuit(
    logical_state="0",
    error_qubit=None,
    pauli_error=None,
    n_rounds=1
):
    """English documentation for this computational helper."""

    n_data = 9
    n_stabilizers = len(SHOR_STABILIZERS)
    n_ancillas = n_stabilizers
    n_classical = n_rounds * n_stabilizers

    qc = QuantumCircuit(n_data + n_ancillas, n_classical)

    encode_shor_block(
        qc,
        offset=0,
        logical_state=logical_state
    )

    if error_qubit is not None:
        inject_pauli_error(qc, error_qubit, pauli_error)

    for r in range(n_rounds):
        for s, (_, pauli, data_qubits) in enumerate(SHOR_STABILIZERS):
            ancilla = n_data + s
            cbit = r * n_stabilizers + s

            measure_pauli_product(
                qc,
                data_qubits=data_qubits,
                pauli=pauli,
                ancilla=ancilla,
                classical_bit_index=cbit
            )

        if r < n_rounds - 1:
            for ancilla in shor_ancilla_qubits():
                qc.reset(ancilla)

        qc.barrier()

    return qc


def interpret_shor_syndrome(syndrome):
    """English documentation for this computational helper."""

    if syndrome is None:
        return "syndrome indefinida"

    if len(syndrome) != 8:
        return "English diagnostic label"

    z_part = syndrome[:6]
    x_part = syndrome[6:]

    bit_flip_errors = []

    triples = [
        (z_part[0:2], (0, 1, 2)),
        (z_part[2:4], (3, 4, 5)),
        (z_part[4:6], (6, 7, 8)),
    ]

    for pair, qubits in triples:
        q0, q1, q2 = qubits

        if pair == "10":
            bit_flip_errors.append(q0)
        elif pair == "11":
            bit_flip_errors.append(q1)
        elif pair == "01":
            bit_flip_errors.append(q2)

    if x_part == "00":
        phase_block = None
    elif x_part == "10":
        phase_block = "English diagnostic label"
    elif x_part == "11":
        phase_block = "English diagnostic label"
    elif x_part == "01":
        phase_block = "English diagnostic label"
    else:
        phase_block = "indeterminado"

    messages = []

    if not bit_flip_errors and phase_block is None:
        messages.append("English diagnostic label")

    if bit_flip_errors:
        messages.append(f"English diagnostic label")

    if phase_block is not None:
        messages.append(f"English diagnostic label")

    return "; ".join(messages)


# ============================================================
# Explanation translated; consult the archived notebook for the original wording.
# ============================================================

def sanitize_filename(text):
    """English documentation for this computational helper."""

    output = text.lower()
    output = output.replace(" ", "_")
    output = output.replace("—", "")
    output = output.replace(":", "")
    output = output.replace("English diagnostic label", "c")
    output = output.replace("English diagnostic label", "a")
    output = output.replace("English diagnostic label", "a")
    output = output.replace("English diagnostic label", "e")
    output = output.replace("English diagnostic label", "i")
    output = output.replace("English diagnostic label", "o")
    output = output.replace("English diagnostic label", "u")

    return output


def run_repetition_experiment(
    title,
    logical_state="0",
    error_qubit=None,
    pauli_error=None,
    readout_error_probability=READOUT_ERROR_PROBABILITY,
    n_rounds=N_ROUNDS,
    shots=SHOTS
):
    """English documentation for this computational helper."""

    n_stabilizers = 2

    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

    # --------------------------------------------------------
    # Explanation translated; consult the archived notebook for the original wording.
    # --------------------------------------------------------

    qc_single = build_repetition_syndrome_circuit(
        logical_state=logical_state,
        error_qubit=error_qubit,
        pauli_error=pauli_error,
        n_rounds=1
    )

    counts_single_ideal = simulate_counts(
        qc_single,
        shots=shots,
        noise_model=None
    )

    syndrome_single_ideal = single_round_syndrome_counts(
        counts_single_ideal,
        n_stabilizers=n_stabilizers,
        round_index=0
    )

    # --------------------------------------------------------
    # Explanation translated; consult the archived notebook for the original wording.
    # --------------------------------------------------------

    noise_single = build_ancilla_readout_noise_model(
        ancilla_qubits=repetition_ancilla_qubits(),
        p_readout=readout_error_probability
    )

    counts_single_noisy = simulate_counts(
        qc_single,
        shots=shots,
        noise_model=noise_single
    )

    syndrome_single_noisy = single_round_syndrome_counts(
        counts_single_noisy,
        n_stabilizers=n_stabilizers,
        round_index=0
    )

    # --------------------------------------------------------
    # Explanation translated; consult the archived notebook for the original wording.
    # --------------------------------------------------------

    qc_repeated = build_repetition_syndrome_circuit(
        logical_state=logical_state,
        error_qubit=error_qubit,
        pauli_error=pauli_error,
        n_rounds=n_rounds
    )

    noise_repeated = build_ancilla_readout_noise_model(
        ancilla_qubits=repetition_ancilla_qubits(),
        p_readout=readout_error_probability
    )

    counts_repeated_noisy = simulate_counts(
        qc_repeated,
        shots=shots,
        noise_model=noise_repeated
    )

    syndrome_repeated_majority = majority_syndrome_counts(
        counts_repeated_noisy,
        n_stabilizers=n_stabilizers,
        n_rounds=n_rounds
    )

    # --------------------------------------------------------
    # Explanation translated; consult the archived notebook for the original wording.
    # --------------------------------------------------------

    print_distribution(
        "English diagnostic label",
        syndrome_single_ideal
    )

    print_distribution(
        "English diagnostic label",
        syndrome_single_noisy
    )

    print_distribution(
        f"English diagnostic label",
        syndrome_repeated_majority
    )

    best_ideal, _ = most_likely(syndrome_single_ideal)
    best_noisy, _ = most_likely(syndrome_single_noisy)
    best_majority, _ = most_likely(syndrome_repeated_majority)

    print("English diagnostic label")
    print(f"English diagnostic label")
    print(f"English diagnostic label")
    print(f"Measurement repetida ruidosa: {best_majority} -> {interpret_repetition_syndrome(best_majority)}")

    # --------------------------------------------------------
    # Explanation translated; consult the archived notebook for the original wording.
    # --------------------------------------------------------

    if MAKE_PLOTS:
        safe_title = sanitize_filename(title)

        plot_distribution(
            syndrome_single_ideal,
            title=f"English diagnostic label",
            filename=f"{safe_title}_single_ideal.png"
        )

        plot_distribution(
            syndrome_single_noisy,
            title=f"English diagnostic label",
            filename=f"{safe_title}_single_noisy.png"
        )

        plot_distribution(
            syndrome_repeated_majority,
            title=f"English diagnostic label",
            filename=f"{safe_title}_majority.png"
        )

    return {
        "qc_single": qc_single,
        "qc_repeated": qc_repeated,
        "single_ideal": syndrome_single_ideal,
        "single_noisy": syndrome_single_noisy,
        "repeated_majority": syndrome_repeated_majority,
    }


def run_shor_experiment(
    title,
    logical_state="0",
    error_qubit=None,
    pauli_error=None,
    readout_error_probability=READOUT_ERROR_PROBABILITY,
    n_rounds=N_ROUNDS,
    shots=SHOTS
):
    """English documentation for this computational helper."""

    n_stabilizers = len(SHOR_STABILIZERS)

    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

    print("\nOrdem da syndrome de Shor:")

    for i, (name, _, _) in enumerate(SHOR_STABILIZERS):
        print(f"s{i} = {name}")

    # --------------------------------------------------------
    # Explanation translated; consult the archived notebook for the original wording.
    # --------------------------------------------------------

    qc_single = build_shor_syndrome_circuit(
        logical_state=logical_state,
        error_qubit=error_qubit,
        pauli_error=pauli_error,
        n_rounds=1
    )

    counts_single_ideal = simulate_counts(
        qc_single,
        shots=shots,
        noise_model=None
    )

    syndrome_single_ideal = single_round_syndrome_counts(
        counts_single_ideal,
        n_stabilizers=n_stabilizers,
        round_index=0
    )

    # --------------------------------------------------------
    # Explanation translated; consult the archived notebook for the original wording.
    # --------------------------------------------------------

    noise_single = build_ancilla_readout_noise_model(
        ancilla_qubits=shor_ancilla_qubits(),
        p_readout=readout_error_probability
    )

    counts_single_noisy = simulate_counts(
        qc_single,
        shots=shots,
        noise_model=noise_single
    )

    syndrome_single_noisy = single_round_syndrome_counts(
        counts_single_noisy,
        n_stabilizers=n_stabilizers,
        round_index=0
    )

    # --------------------------------------------------------
    # Explanation translated; consult the archived notebook for the original wording.
    # --------------------------------------------------------

    qc_repeated = build_shor_syndrome_circuit(
        logical_state=logical_state,
        error_qubit=error_qubit,
        pauli_error=pauli_error,
        n_rounds=n_rounds
    )

    noise_repeated = build_ancilla_readout_noise_model(
        ancilla_qubits=shor_ancilla_qubits(),
        p_readout=readout_error_probability
    )

    counts_repeated_noisy = simulate_counts(
        qc_repeated,
        shots=shots,
        noise_model=noise_repeated
    )

    syndrome_repeated_majority = majority_syndrome_counts(
        counts_repeated_noisy,
        n_stabilizers=n_stabilizers,
        n_rounds=n_rounds
    )

    # --------------------------------------------------------
    # Explanation translated; consult the archived notebook for the original wording.
    # --------------------------------------------------------

    print_distribution(
        "English diagnostic label",
        syndrome_single_ideal
    )

    print_distribution(
        "English diagnostic label",
        syndrome_single_noisy
    )

    print_distribution(
        f"English diagnostic label",
        syndrome_repeated_majority
    )

    best_ideal, _ = most_likely(syndrome_single_ideal)
    best_noisy, _ = most_likely(syndrome_single_noisy)
    best_majority, _ = most_likely(syndrome_repeated_majority)

    print("English diagnostic label")
    print(f"English diagnostic label")
    print(f"English diagnostic label")
    print(f"Measurement repetida ruidosa: {best_majority} -> {interpret_shor_syndrome(best_majority)}")

    # --------------------------------------------------------
    # Explanation translated; consult the archived notebook for the original wording.
    # --------------------------------------------------------

    if MAKE_PLOTS:
        safe_title = sanitize_filename(title)

        plot_distribution(
            syndrome_single_ideal,
            title=f"English diagnostic label",
            filename=f"{safe_title}_single_ideal.png"
        )

        plot_distribution(
            syndrome_single_noisy,
            title=f"English diagnostic label",
            filename=f"{safe_title}_single_noisy.png"
        )

        plot_distribution(
            syndrome_repeated_majority,
            title=f"English diagnostic label",
            filename=f"{safe_title}_majority.png"
        )

    return {
        "qc_single": qc_single,
        "qc_repeated": qc_repeated,
        "single_ideal": syndrome_single_ideal,
        "single_noisy": syndrome_single_noisy,
        "repeated_majority": syndrome_repeated_majority,
    }


# ============================================================
# Explanation translated; consult the archived notebook for the original wording.
# ============================================================

print("\n============================================================")
print("English diagnostic label")
print("============================================================")

print(f"\nShots: {SHOTS}")
print(f"English diagnostic label")
print(f"Probabilidade de erro de leitura nas ancillas: {READOUT_ERROR_PROBABILITY}")


# ============================================================
# Explanation translated; consult the archived notebook for the original wording.
# ============================================================

result_rep_no_error = run_repetition_experiment(
    title="English diagnostic label",
    logical_state="0",
    error_qubit=None,
    pauli_error=None
)

result_rep_x_q0 = run_repetition_experiment(
    title="English diagnostic label",
    logical_state="0",
    error_qubit=0,
    pauli_error="X"
)

result_rep_x_q1 = run_repetition_experiment(
    title="English diagnostic label",
    logical_state="0",
    error_qubit=1,
    pauli_error="X"
)

result_rep_x_q2 = run_repetition_experiment(
    title="English diagnostic label",
    logical_state="0",
    error_qubit=2,
    pauli_error="X"
)


# ============================================================
# Explanation translated; consult the archived notebook for the original wording.
# ============================================================

result_shor_no_error = run_shor_experiment(
    title="Shor code — sem erro",
    logical_state="0",
    error_qubit=None,
    pauli_error=None
)

result_shor_x_q4 = run_shor_experiment(
    title="Shor code — erro X em q4",
    logical_state="0",
    error_qubit=4,
    pauli_error="X"
)

result_shor_z_q4 = run_shor_experiment(
    title="Shor code — erro Z em q4",
    logical_state="0",
    error_qubit=4,
    pauli_error="Z"
)

result_shor_y_q4 = run_shor_experiment(
    title="Shor code — erro Y em q4",
    logical_state="0",
    error_qubit=4,
    pauli_error="Y"
)


# ============================================================
# Explanation translated; consult the archived notebook for the original wording.
# ============================================================

print("\n============================================================")
print("English diagnostic label")
print("============================================================")

qc_rep_example = build_repetition_syndrome_circuit(
    logical_state="0",
    error_qubit=1,
    pauli_error="X",
    n_rounds=1
)

print(qc_rep_example.draw(output="text"))


print("\n============================================================")
print("English diagnostic label")
print("============================================================")

qc_shor_example = build_shor_syndrome_circuit(
    logical_state="0",
    error_qubit=4,
    pauli_error="X",
    n_rounds=N_ROUNDS
)

print(f"English diagnostic label")
print(f"English diagnostic label")

print("English diagnostic label")
print("English diagnostic label")
print("English diagnostic label")


print("\n============================================================")
print("FIM DO PASSO 3")
print("============================================================")